# Propensity Score Model Evaluation

Compare two feature schemes for predicting ICI receipt:
1. **Covariates only**: AGE + GENDER + CANCER_TYPE + LINE + PANEL_VERSION
2. **Covariates + Embeddings**: AGE + GENDER + CANCER_TYPE + LINE + PANEL_VERSION + EMBEDDINGS

Both use elastic-net CV logistic regression with 5-fold held-out scoring.

Run separately for each cohort:
- **Cohort 1**: First-line ICI vs all never-ICI (unmatched)
- **Cohort 2**: Lines 1-3, 1:1 matched on (cancer_type, line_category)

Evaluation levels:
- Pan-cancer (all patients)
- Within cancer type
- Within line category

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings('ignore', category=ConvergenceWarning)

from biomarker_common import DATA_PATH, SURV_PATH, load_note_embeddings
from embed_surv_utils import generate_survival_embedding_df

COHORTS = ['cohort1', 'cohort2']
BUFFER = 30  # day buffer for note embeddings
SEED = 1234

## 1. Load shared data (embeddings, demographics, cancer type, panel version)

In [ ]:
# Shared data loaded once, then filtered per cohort
notes_meta, embeddings_data = load_note_embeddings()
note_types = ['Clinician', 'Imaging', 'Pathology']
pool_fx = {nt: 'time_decay_mean' for nt in note_types}

surv_df = pd.read_csv(os.path.join(SURV_PATH, 'death_met_surv_df.csv'),
                       usecols=['DFCI_MRN', 'GENDER', 'AGE_AT_TREATMENTSTART'])
surv_df = surv_df.drop_duplicates('DFCI_MRN')

cancer_type_df = pd.read_csv(
    os.path.join(DATA_PATH, 'clinical_and_genomic_features/cancer_type_df.csv'))
cancer_type_cols = [c for c in cancer_type_df.columns if c.startswith('CANCER_TYPE_')]

COHORT_PATH = os.path.join(DATA_PATH, 'biomarker_analysis/matched_cohorts/')
print("Shared data loaded.")

In [ ]:
def load_cohort_with_features(cohort_name):
    """Load a cohort and merge demographics, cancer type, and embeddings."""
    cdf = pd.read_csv(os.path.join(COHORT_PATH, f'matched_cohort_{cohort_name}.csv'))
    cdf['treatment_start_date'] = pd.to_datetime(cdf['treatment_start_date'])
    
    # Demographics
    cdf = cdf.merge(surv_df, on='DFCI_MRN', how='left')
    
    # Cancer type dummies
    cdf = cdf.merge(cancer_type_df[['DFCI_MRN'] + cancer_type_cols], on='DFCI_MRN', how='left')
    
    # Line dummies — cohort1 is first-line ICI only, no line variation to adjust for
    if cohort_name != 'cohort1':
        cdf['line_category'] = cdf['line_category'].clip(upper=3)
        cdf = pd.get_dummies(cdf, columns=['line_category'], prefix='LINE', drop_first=True, dtype=int)
        line_cols = [c for c in cdf.columns if c.startswith('LINE_')]
    else:
        line_cols = []
    
    # Embeddings
    notes_sub = (
        notes_meta[notes_meta['DFCI_MRN'].isin(cdf['DFCI_MRN'])]
        .merge(cdf[['DFCI_MRN', 'treatment_start_date']].drop_duplicates('DFCI_MRN'),
               on='DFCI_MRN', how='left')
        .assign(NOTE_TIME_REL=lambda df: (
            pd.to_datetime(df['NOTE_DATETIME']) - df['treatment_start_date']).dt.days)
    )
    embedding_vals = generate_survival_embedding_df(
        notes_meta=notes_sub, survival_df=None, embedding_array=embeddings_data,
        note_types=note_types, note_timing_col='NOTE_TIME_REL',
        max_note_window=-BUFFER, pool_fx=pool_fx, decay_param=0.01, continuous_window=False)
    embedding_cols = [c for c in embedding_vals.columns if c != 'DFCI_MRN']
    cdf = cdf.merge(embedding_vals.dropna(), on='DFCI_MRN')
    
    # Define feature schemes (matches ICI_LRs.py: no panel version in PS model)
    covariate_cols = ['AGE_AT_TREATMENTSTART', 'GENDER'] + cancer_type_cols + line_cols
    schemes = {
        'covariates_only': covariate_cols,
        'covariates_plus_embeddings': covariate_cols + embedding_cols,
    }
    
    n_ici = int(cdf['PX_on_ICI'].sum())
    n_ctrl = len(cdf) - n_ici
    print(f"[{cohort_name}] {n_ici} ICI + {n_ctrl} controls = {len(cdf)}, "
          f"embedding dims: {len(embedding_cols)}, "
          f"covariate dims: {len(covariate_cols)}, "
          f"line dummies: {line_cols}")
    
    return cdf, schemes, embedding_cols

## 2. Train propensity models for each cohort

In [ ]:
def train_propensity_cv(df, feature_cols, label, n_splits=5, seed=SEED):
    """Elastic-net CV LR with held-out propensity scores."""
    X = df[feature_cols].values
    y = df['PX_on_ICI'].astype(int).values
    mrns = df['DFCI_MRN'].values
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out_mrns, out_probs, out_true = [], [], []
    fold_info = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        scaler = StandardScaler().fit(X[train_idx])
        X_train_s = scaler.transform(X[train_idx])
        X_test_s = scaler.transform(X[test_idx])
        
        clf = LogisticRegressionCV(
            penalty='elasticnet', solver='saga',
            Cs=8, l1_ratios=[0.1, 0.5, 0.9],
            cv=3, scoring='roc_auc',
            max_iter=2000, tol=1e-3,
            random_state=seed, n_jobs=4,
        )
        clf.fit(X_train_s, y[train_idx])
        
        probs = clf.predict_proba(X_test_s)[:, 1]
        out_mrns.extend(mrns[test_idx])
        out_probs.extend(probs)
        out_true.extend(y[test_idx])
        
        n_nz = np.sum(clf.coef_ != 0)
        fold_info.append(f"fold {fold}: C={clf.C_[0]:.4f}, l1={clf.l1_ratio_[0]:.2f}, feats={n_nz}/{clf.coef_.size}")
    
    auc = roc_auc_score(out_true, out_probs)
    print(f"  [{label}] AUC={auc:.4f}")
    for fi in fold_info:
        print(f"    {fi}")
    
    return pd.DataFrame({'DFCI_MRN': out_mrns, 'ps': out_probs, 'ICI': out_true})

In [ ]:
# Load cohorts and train models
# Use common patient set across both PS schemes (matches ICI_LRs.py)
cohort_data = {}   # cohort_name -> (df, schemes, embedding_cols)
all_results = {}   # (cohort_name, scheme_name) -> scores_df

for cohort_name in COHORTS:
    print(f"\n{'='*70}")
    print(f"COHORT: {cohort_name}")
    print(f"{'='*70}")
    
    cdf, schemes, emb_cols = load_cohort_with_features(cohort_name)
    cohort_data[cohort_name] = (cdf, schemes, emb_cols)
    
    # Restrict to patients with complete data for the most demanding feature set
    # (covariates + embeddings), so both schemes use identical patients.
    all_feat_cols = schemes['covariates_plus_embeddings']
    common_df = cdf.dropna(subset=all_feat_cols)
    print(f"  Common patient set: {len(common_df)} "
          f"(dropped {len(cdf) - len(common_df)} with missing data)")
    
    for scheme_name, feat_cols in schemes.items():
        # Remove features that are all-zero in this cohort
        active_feats = [c for c in feat_cols if common_df[c].abs().sum() > 0]
        print(f"\n  {scheme_name}: {len(common_df)} patients, {len(active_feats)} active features")
        scores = train_propensity_cv(common_df, active_feats, f"{cohort_name}/{scheme_name}")
        
        # Merge back cohort labels for stratified evaluation
        cohort_labels = pd.read_csv(
            os.path.join(COHORT_PATH, f'matched_cohort_{cohort_name}.csv'),
            usecols=['DFCI_MRN', 'cancer_type', 'line_category'])
        scores = scores.merge(cohort_labels, on='DFCI_MRN', how='left')
        all_results[(cohort_name, scheme_name)] = scores

In [ ]:
def eval_auc(df, group_col=None, min_per_class=20):
    """Compute AUC + Brier score overall or within groups."""
    if group_col is None:
        auc = roc_auc_score(df['ICI'], df['ps'])
        brier = brier_score_loss(df['ICI'], df['ps'])
        return pd.DataFrame([{'group': 'overall', 'n': len(df),
                              'n_ICI': df['ICI'].sum(), 'n_ctrl': (1-df['ICI']).sum(),
                              'AUC': auc, 'Brier': brier}])
    
    rows = []
    for grp, gdf in df.groupby(group_col):
        n_pos, n_neg = gdf['ICI'].sum(), (1 - gdf['ICI']).sum()
        if n_pos < min_per_class or n_neg < min_per_class:
            rows.append({'group': grp, 'n': len(gdf), 'n_ICI': n_pos,
                         'n_ctrl': n_neg, 'AUC': np.nan, 'Brier': np.nan})
            continue
        auc = roc_auc_score(gdf['ICI'], gdf['ps'])
        brier = brier_score_loss(gdf['ICI'], gdf['ps'])
        rows.append({'group': grp, 'n': len(gdf), 'n_ICI': n_pos,
                     'n_ctrl': n_neg, 'AUC': auc, 'Brier': brier})
    return pd.DataFrame(rows).sort_values('n', ascending=False)

## 3. Pan-cancer evaluation + ROC curves

In [ ]:
# Pan-cancer AUC table
pan_rows = []
for (cohort_name, scheme_name), df in all_results.items():
    row = eval_auc(df)
    row['cohort'] = cohort_name
    row['scheme'] = scheme_name
    pan_rows.append(row)
pan_cancer = pd.concat(pan_rows, ignore_index=True)
print("Pan-cancer AUC:")
display(pan_cancer[['cohort', 'scheme', 'n', 'n_ICI', 'n_ctrl', 'AUC', 'Brier']])

# ROC curves — one subplot per cohort
fig, axes = plt.subplots(1, len(COHORTS), figsize=(6 * len(COHORTS), 5))
if len(COHORTS) == 1:
    axes = [axes]

for ax, cohort_name in zip(axes, COHORTS):
    for scheme_name in ['covariates_only', 'covariates_plus_embeddings']:
        df = all_results[(cohort_name, scheme_name)]
        fpr, tpr, _ = roc_curve(df['ICI'], df['ps'])
        auc = roc_auc_score(df['ICI'], df['ps'])
        ax.plot(fpr, tpr, label=f'{scheme_name} (AUC={auc:.3f})')
    ax.plot([0, 1], [0, 1], '--', color='gray')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'{cohort_name}')
    ax.legend(loc='lower right')

plt.suptitle('Pan-cancer ROC: ICI receipt prediction', y=1.02)
plt.tight_layout()
plt.show()

## 4. Within cancer type

In [ ]:
for cohort_name in COHORTS:
    print(f"\n{'='*60}")
    print(f"COHORT: {cohort_name}")
    print(f"{'='*60}")
    for scheme_name in ['covariates_only', 'covariates_plus_embeddings']:
        df = all_results[(cohort_name, scheme_name)]
        print(f"\n--- {scheme_name} ---")
        display(eval_auc(df, 'cancer_type'))

In [ ]:
# Side-by-side AUC comparison by cancer type — one plot per cohort
fig, axes = plt.subplots(1, len(COHORTS), figsize=(10 * len(COHORTS), 6))
if len(COHORTS) == 1:
    axes = [axes]

for ax, cohort_name in zip(axes, COHORTS):
    ct_aucs = []
    for scheme_name in ['covariates_only', 'covariates_plus_embeddings']:
        df = all_results[(cohort_name, scheme_name)]
        ea = eval_auc(df, 'cancer_type')
        ea['scheme'] = scheme_name
        ct_aucs.append(ea)
    ct_aucs = pd.concat(ct_aucs).dropna(subset=['AUC'])
    ct_pivot = ct_aucs.pivot(index='group', columns='scheme', values='AUC').sort_values(
        'covariates_plus_embeddings', ascending=True)
    ct_pivot.plot.barh(ax=ax)
    ax.set_xlabel('AUC'); ax.set_ylabel('')
    ax.set_title(f'{cohort_name}: AUC by cancer type')
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax.legend(title='Scheme')

plt.tight_layout()
plt.show()

In [ ]:
# Within line category (mainly relevant for cohort2)
for cohort_name in COHORTS:
    print(f"\n{'='*60}")
    print(f"COHORT: {cohort_name}")
    print(f"{'='*60}")
    for scheme_name in ['covariates_only', 'covariates_plus_embeddings']:
        df = all_results[(cohort_name, scheme_name)]
        print(f"\n--- {scheme_name} ---")
        display(eval_auc(df, 'line_category'))

## 5. PS distribution + calibration diagnostics

In [ ]:
# PS distributions by treatment arm — one row per cohort
scheme_names = ['covariates_only', 'covariates_plus_embeddings']
fig, axes = plt.subplots(len(COHORTS), len(scheme_names),
                          figsize=(6 * len(scheme_names), 4 * len(COHORTS)))
if len(COHORTS) == 1:
    axes = [axes]

for i, cohort_name in enumerate(COHORTS):
    for j, scheme_name in enumerate(scheme_names):
        ax = axes[i][j]
        df = all_results[(cohort_name, scheme_name)]
        for label, grp in df.groupby('ICI'):
            tag = 'ICI' if label == 1 else 'Control'
            ax.hist(grp['ps'], bins=50, alpha=0.5, label=tag, density=True)
        ax.set_title(f'{cohort_name} / {scheme_name}')
        ax.set_xlabel('Propensity Score')
        ax.set_ylabel('Density')
        ax.legend()

plt.suptitle('PS distributions by treatment arm', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Calibration plots
fig, axes = plt.subplots(len(COHORTS), len(scheme_names),
                          figsize=(6 * len(scheme_names), 4 * len(COHORTS)))
if len(COHORTS) == 1:
    axes = [axes]

for i, cohort_name in enumerate(COHORTS):
    for j, scheme_name in enumerate(scheme_names):
        ax = axes[i][j]
        df = all_results[(cohort_name, scheme_name)].copy()
        df['decile'] = pd.qcut(df['ps'], 10, labels=False, duplicates='drop')
        cal = df.groupby('decile').agg(
            mean_ps=('ps', 'mean'), actual_rate=('ICI', 'mean')).reset_index()
        ax.plot(cal['mean_ps'], cal['actual_rate'], 'o-', label='model')
        ax.plot([0, 1], [0, 1], '--', color='gray', label='perfect')
        ax.set_xlabel('Mean predicted P(ICI)')
        ax.set_ylabel('Actual ICI rate')
        ax.set_title(f'{cohort_name} / {scheme_name}')
        ax.legend()

plt.tight_layout()
plt.show()

## 6. Summary table + save

In [ ]:
summary_rows = []

for (cohort_name, scheme_name), df in all_results.items():
    # Pan-cancer
    summary_rows.append({
        'cohort': cohort_name, 'scheme': scheme_name,
        'level': 'pan-cancer', 'group': 'all',
        'n': len(df), 'AUC': roc_auc_score(df['ICI'], df['ps']),
        'Brier': brier_score_loss(df['ICI'], df['ps']),
    })
    # By cancer type
    for grp, gdf in df.groupby('cancer_type'):
        if gdf['ICI'].sum() >= 20 and (1 - gdf['ICI']).sum() >= 20:
            summary_rows.append({
                'cohort': cohort_name, 'scheme': scheme_name,
                'level': 'cancer_type', 'group': grp,
                'n': len(gdf), 'AUC': roc_auc_score(gdf['ICI'], gdf['ps']),
                'Brier': brier_score_loss(gdf['ICI'], gdf['ps']),
            })
    # By line
    for grp, gdf in df.groupby('line_category'):
        if gdf['ICI'].sum() >= 20 and (1 - gdf['ICI']).sum() >= 20:
            summary_rows.append({
                'cohort': cohort_name, 'scheme': scheme_name,
                'level': 'line', 'group': grp,
                'n': len(gdf), 'AUC': roc_auc_score(gdf['ICI'], gdf['ps']),
                'Brier': brier_score_loss(gdf['ICI'], gdf['ps']),
            })

summary_df = pd.DataFrame(summary_rows).round({'AUC': 4, 'Brier': 4})
display(summary_df)

# Save
OUT_DIR = os.path.join(DATA_PATH, 'treatment_prediction/ps_evaluation/')
os.makedirs(OUT_DIR, exist_ok=True)
summary_df.to_csv(os.path.join(OUT_DIR, 'ps_evaluation_summary.csv'), index=False)

for (cohort_name, scheme_name), df in all_results.items():
    df.to_csv(os.path.join(OUT_DIR, f'held_out_scores_{cohort_name}_{scheme_name}.csv'), index=False)
    print(f"Saved {cohort_name}/{scheme_name}: {len(df)} patients")

print(f"\nAll saved to {OUT_DIR}")